# Architecture Comparison: Label Noise Sweep Validation

This notebook validates the merged `uq_classification` / `uq_benchmarks` pipeline against the paper-style
label noise experiment.

It is intentionally wired to the current `scripts/run_validation_experiments.py` output contract:
- load flat `metrics.csv` files
- surface warnings instead of silently plotting blanks
- call the central runner when `RUN_NEW_EXPERIMENTS = True`


In [ ]:
from pathlib import Path
import sys

_PROJECT_ROOT = Path(__file__).resolve().parents[2]
if str(_PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from walaris.notebook_support import (
    ARCHITECTURE_STYLES,
    ensure_columns,
    find_project_root,
    get_row3_signals,
    load_sweep_metrics,
    plot_individual_signals,
    plot_method_uncertainty_comparison,
    present_architectures,
    print_method_architecture_mapping,
    run_validation_experiments,
    summarize_trends,
    validation_dir,
)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

project_root = find_project_root()
results_root = validation_dir(project_root) / "label_noise_sweep"
print(f"Project root: {project_root}")
print(f"Results directory: {results_root}")
print(f"Python: {sys.executable}")


## Run Or Load

Set `RUN_NEW_EXPERIMENTS = True` only when your notebook kernel has the full ML environment.
The notebook delegates reruns to `scripts/run_validation_experiments.py --sweep label_noise` so it stays
aligned with the maintained pipeline.


In [ ]:
RUN_NEW_EXPERIMENTS = False
VALIDATION_MODE = "full"

if RUN_NEW_EXPERIMENTS:
    result = run_validation_experiments(
        project_root=project_root,
        sweep="label_noise",
        mode=VALIDATION_MODE,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Validation runner failed.")


In [ ]:
loaded = load_sweep_metrics("label_noise", project_root=project_root)
df_metrics = loaded.df.copy()

for warning in loaded.warnings:
    print(f"WARNING: {warning}")

print(f"Loaded {len(df_metrics)} rows")
display(
    ensure_columns(
        df_metrics,
        [
            "architecture",
            "noise_percent",
            "accuracy",
            "mean_epistemic_uncertainty",
            "mean_aleatoric_uncertainty",
            "mean_total_uncertainty",
            "noise_rate",
            "noise_percent",
            "experiment_name",
        ],
    )
)

if df_metrics.empty:
    raise RuntimeError("No validation rows were loaded.")


## Paper Alignment Notes

- This notebook checks whether the expected **aleatoric** trend is visible in the merged pipeline.
- If you see a warning about legacy label-noise scaling, do **not** use the old results for a thesis figure.
- Empty plots should now indicate genuinely missing data, not silent schema mismatches.


In [ ]:
def plot_dual_axis(df, left_metric="accuracy", right_metric="mean_aleatoric_uncertainty", title="Architecture Comparison: Label Noise Sweep Validation"):
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax2 = ax1.twinx()

    ax1.set_xlabel("Label Noise (%)")
    ax1.set_ylabel(left_metric.replace("_", " ").title(), color="tab:blue")
    ax2.set_ylabel(right_metric.replace("_", " ").title(), color="tab:red")

    for architecture, style in ARCHITECTURE_STYLES.items():
        arch_df = df[df["architecture"] == architecture].sort_values("noise_percent")
        if arch_df.empty:
            continue

        ax1.plot(
            arch_df["noise_percent"],
            arch_df[left_metric],
            color=style["color"],
            marker=style["marker"],
            linewidth=2,
            label=f"{architecture} / {left_metric}",
        )
        ax2.plot(
            arch_df["noise_percent"],
            arch_df[right_metric],
            color=style["color"],
            marker=style["marker"],
            linestyle="--",
            linewidth=2,
            alpha=0.8,
            label=f"{architecture} / {right_metric}",
        )

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="center left", bbox_to_anchor=(1.15, 0.5))
    plt.title(title)
    plt.tight_layout()
    plt.show()


plot_dual_axis(df_metrics)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plots = [
    ("accuracy", axes[0, 0]),
    ("mean_epistemic_uncertainty", axes[0, 1]),
    ("mean_aleatoric_uncertainty", axes[1, 0]),
    ("mean_total_uncertainty", axes[1, 1]),
]

for metric, ax in plots:
    for architecture, style in ARCHITECTURE_STYLES.items():
        arch_df = df_metrics[df_metrics["architecture"] == architecture].sort_values("noise_percent")
        if arch_df.empty:
            continue
        ax.plot(
            arch_df["noise_percent"],
            arch_df[metric],
            color=style["color"],
            marker=style["marker"],
            linewidth=2,
            label=architecture,
        )

    ax.set_title(metric.replace("_", " ").title())
    ax.set_xlabel("Label Noise (%)")
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


## Individual Signal AUROC Grid

Epistemic and aleatoric AUROC are plotted separately (do not average them).


In [ ]:
plot_individual_signals(df_metrics, sweep_type="label_noise")


## Method Uncertainty Comparison

Layout adapts to the loaded data:
- **Architecture rows:** One row per disentanglement type present in the metrics
  (e.g., PyTorch Attribution when only your validation runs are loaded; plus
  Gaussian Logits / Information Theoretic when paper Keras rows are present).
  Columns = selected architectures.
- **Signal AUROC row (last):** Top 4 candidate signals ranked by sweep-appropriate
  mean AUROC (epistemic for dataset-size, aleatoric for label-noise).  Each column
  shows per-architecture AUROC lines for that signal — distinct per column.

The cell below prints the **architecture list** and **resolved mapping**, then plots.


In [ ]:
archs = present_architectures(df_metrics)
print("Architectures in data:", archs)
mapping = print_method_architecture_mapping(df_metrics)

row3_signals = get_row3_signals(df_metrics, sweep_type="label_noise")
print("\nSignal AUROC row (top 4 by mean AUROC):")
for signal, auroc in row3_signals:
    print(f"  {signal}: {auroc:.4f}")

plot_method_uncertainty_comparison(df_metrics, sweep_type="label_noise", architectures=archs)


In [ ]:
trend_summary = summarize_trends(df_metrics, "label_noise")
display(trend_summary)

trend_path = results_root / "trend_summary.csv"
trend_summary.to_csv(trend_path, index=False)
print(f"Saved trend summary to: {trend_path}")
